# NaturalisticDiffInt — 02: Naturalistic Pipeline

**Goal:** Run the NMPH MemoryArchive on real StudyForrest data:
1. Load pre-extracted VGG-19 feature timeseries from Google Drive
2. Load GSBS event boundaries (or fall back to fixed-window segmentation)
3. PCA category/item split (Decision D6)
4. Run MemoryArchive: encode event traces, find pairmates, run competition
5. Compute RSA change matrices; save for brain comparison (notebook 03)

**Expected Google Drive layout:**
```
MyDrive/NaturalisticDiffInt/
    features/
        vgg19_pool5_run{1-8}.npy     # (n_trs, 4096) float32
    boundaries/
        gsbs_boundaries_run{1-8}.npy # (n_events,) int TR onsets   [optional]
        gsbs_durations_run{1-8}.npy  # (n_events,) int TR durations [optional]
```
If boundaries are missing, fixed-window segmentation is used as fallback.


## 0. GitHub Sync — Setup

In [ ]:
from google.colab import userdata
import os, subprocess

GITHUB_USER  = "drgzkr"
GITHUB_REPO  = "NaturalisticDiffInt"
REPO_PATH    = f"/content/{GITHUB_REPO}"
NOTEBOOK_REL = "notebooks/analysis/02_naturalistic_pipeline.ipynb"

_token  = userdata.get("GITHUB_TOKEN")
_remote = f"https://{_token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

subprocess.run(["git", "config", "--global", "user.name", "Colab"], check=True)
subprocess.run(["git", "config", "--global", "user.email", "colab@naturalistic-diffint.local"], check=True)

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", _remote, REPO_PATH], check=True)
    print(f"Cloned  -> {REPO_PATH}")
else:
    subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", _remote])
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
    print(f"Pulled  -> {REPO_PATH}")

del _token, _remote
print(f"Repo ready at {REPO_PATH}")


## 1. Mount Google Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = "/content/drive/MyDrive/NaturalisticDiffInt"
FEAT_DIR    = f"{DRIVE_ROOT}/features"
BOUND_DIR   = f"{DRIVE_ROOT}/boundaries"
RESULTS_DIR = f"{DRIVE_ROOT}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

RUNS = list(range(1, 9))

for run in RUNS:
    fpath = f"{FEAT_DIR}/vgg19_pool5_run{run}.npy"
    print(f"Run {run} features: {'OK' if os.path.exists(fpath) else 'MISSING'}")


## 2. Imports and hyperparameters

In [ ]:
import sys
sys.path.insert(0, REPO_PATH)

import math, pickle, warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import Optional
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

# Hyperparameters (see docs/decisions.md)
N_CATEGORY        = 20
COMPETITOR_THRESH = 0.70
MIN_TEMPORAL_GAP  = 3
OSC_AMP           = 0.11
OSC_PERIOD        = 10
N_OSC_TICKS       = 20
N_HIDDEN          = 64
LR                = 0.01
WINDOW_SIZE       = 15    # fallback fixed-window size in TRs

print("Config OK.")


## 3. Feature loading helpers

In [ ]:
def load_run_features(run):
    feats = np.load(f"{FEAT_DIR}/vgg19_pool5_run{run}.npy").astype(np.float32)
    print(f"  Run {run}: {feats.shape}")
    return feats

def load_boundaries(run, n_trs):
    op = f"{BOUND_DIR}/gsbs_boundaries_run{run}.npy"
    dp = f"{BOUND_DIR}/gsbs_durations_run{run}.npy"
    if os.path.exists(op) and os.path.exists(dp):
        onsets    = np.load(op).astype(int)
        durations = np.load(dp).astype(int)
        print(f"  Run {run}: GSBS — {len(onsets)} events")
    else:
        onsets    = np.arange(0, n_trs - WINDOW_SIZE, WINDOW_SIZE)
        durations = np.full(len(onsets), WINDOW_SIZE)
        print(f"  Run {run}: fixed windows — {len(onsets)} events")
    return onsets, durations

def event_average(feats, onsets, durations):
    ev = np.zeros((len(onsets), feats.shape[1]), dtype=np.float32)
    for i, (s, d) in enumerate(zip(onsets, durations)):
        e = min(int(s + d), feats.shape[0])
        ev[i] = feats[int(s):e].mean(0)
    return ev

print("Loading all runs...")
run_data = {}
for run in RUNS:
    feats = load_run_features(run)
    onsets, durations = load_boundaries(run, feats.shape[0])
    run_data[run] = dict(
        feats=feats,
        event_feats=event_average(feats, onsets, durations),
        onsets=onsets, durations=durations
    )


## 4. PCA feature split

In [ ]:
scaler = StandardScaler()
X_r1   = scaler.fit_transform(run_data[1]['event_feats'])
pca    = PCA(random_state=42).fit(X_r1)

cumvar = np.cumsum(pca.explained_variance_ratio_)
n_95   = np.argmax(cumvar >= 0.95) + 1
print(f"Top {N_CATEGORY} components: {cumvar[N_CATEGORY-1]*100:.1f}% variance  |  95% at {n_95}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(cumvar[:120], lw=2, color='steelblue')
ax.axvline(N_CATEGORY, color='red', ls='--', lw=1.5, label=f'N_CATEGORY={N_CATEGORY}')
ax.axhline(0.95, color='grey', ls=':', lw=1, label='95%')
ax.set_xlabel("PCA components"); ax.set_ylabel("Cumulative explained variance")
ax.set_title("VGG-19 pool5 PCA — StudyForrest")
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

for run in RUNS:
    Xs  = scaler.transform(run_data[run]['event_feats'])
    proj = pca.transform(Xs)
    cat  = torch.tensor(proj[:, :N_CATEGORY], dtype=torch.float32)
    item = torch.tensor(proj[:, N_CATEGORY:], dtype=torch.float32)
    run_data[run]['cat_feats']  = F.normalize(cat,  dim=1)
    run_data[run]['item_feats'] = F.normalize(item, dim=1)

print("Category/item split done.")


**Interpreting the PCA split:** The explained variance curve shows how much of the VGG-19 feature
variance is captured by successive principal components.

- **Category features** (top `N_CATEGORY` components) capture the *slow-varying, high-variance*
  structure — scene context, dominant objects, rough spatial layout. These should be similar across
  events that share a narrative setting (the naturalistic analogue of "same category" in the lab).
- **Item features** (remaining components) capture *fast-varying, lower-variance* perceptual details —
  colour, texture, exact spatial configuration. These distinguish events that share a setting but
  differ in what is happening moment to moment.

If fewer than ~30–40% of variance is explained by the category features, consider increasing
`N_CATEGORY`. If the cumulative curve is flat after ~10 components, the feature space is
low-dimensional and a smaller N_CATEGORY may be more appropriate.


## 5. MemoryArchive (inline)

In [ ]:
@dataclass
class MemoryTrace:
    idx: int
    hidden: torch.Tensor
    hidden_init: torch.Tensor
    category_features: torch.Tensor
    item_features: torch.Tensor
    n_trs: int = 1; onset_tr: int = 0
    competitor_idxs: list = field(default_factory=list)

    def repr_change(self):
        return F.cosine_similarity(self.hidden.unsqueeze(0), self.hidden_init.unsqueeze(0)).item()


class MemoryArchive:
    def __init__(self, n_hidden=64, competitor_threshold=0.70, min_temporal_gap=3,
                 osc_amp=0.11, osc_period=10, n_osc_ticks=20, lr=0.01):
        self.n_hidden=n_hidden; self.competitor_threshold=competitor_threshold
        self.min_temporal_gap=min_temporal_gap; self.osc_amp=osc_amp
        self.osc_period=osc_period; self.n_osc_ticks=n_osc_ticks; self.lr=lr
        self.traces=[]; self._cp=None; self._ip=None; self.log=[]

    def _init_proj(self, nc, ni):
        self._cp = F.normalize(torch.randn(self.n_hidden, nc), dim=0)
        self._ip = F.normalize(torch.randn(self.n_hidden, ni), dim=0)

    def _project(self, cat, item):
        if self._cp is None: self._init_proj(cat.shape[0], item.shape[0])
        h = self._cp @ cat + self._ip @ item
        return F.normalize(torch.relu(h), dim=0)

    def add_trace(self, cat, item, n_trs=1, onset_tr=0):
        h = self._project(cat, item)
        t = MemoryTrace(len(self.traces), h.clone(), h.clone(), cat.clone(), item.clone(), n_trs, onset_tr)
        self.traces.append(t); return t

    def find_competitors(self, trace):
        out=[]
        for p in self.traces:
            if p.idx >= trace.idx or (trace.idx - p.idx) < self.min_temporal_gap: continue
            sim = F.cosine_similarity(trace.category_features.unsqueeze(0),
                                       p.category_features.unsqueeze(0)).item()
            if sim >= self.competitor_threshold: out.append(p)
        return out

    def _retrieval_pass(self, tgt, comp):
        comp_acts=[]
        for t in range(self.n_osc_ticks):
            osc = 1.0 - self.osc_amp * math.sin(2*math.pi*t/self.osc_period)
            lv  = torch.clamp(torch.dot(tgt.hidden, comp.hidden) / max(osc,1e-6), 0.0, 1.0)
            comp_acts.append(comp.hidden * lv)
        return tgt.hidden.clone(), torch.stack(comp_acts).mean(0)

    def _bcm_step(self, h, pre, post):
        theta = (post**2).mean()
        dw = self.lr * post * (post - theta) * pre
        return F.normalize(torch.clamp(h + dw, 0, 1), dim=0)

    def run_competition(self, trace):
        comps = self.find_competitors(trace)
        trace.competitor_idxs = [c.idx for c in comps]
        results=[]
        for comp in comps:
            sb = F.cosine_similarity(trace.hidden.unsqueeze(0), comp.hidden.unsqueeze(0)).item()
            ta, ca = self._retrieval_pass(trace, comp)
            trace.hidden = self._bcm_step(trace.hidden, ca, ta)
            sa = F.cosine_similarity(trace.hidden.unsqueeze(0), comp.hidden.unsqueeze(0)).item()
            d  = 'differentiation' if sa < sb-0.01 else 'integration' if sa > sb+0.01 else 'no_change'
            e  = dict(target_idx=trace.idx, competitor_idx=comp.idx,
                      sim_before=sb, sim_after=sa, delta_sim=sa-sb,
                      direction=d, competitor_activity=ca.norm().item())
            results.append(e); self.log.append(e)
        return results

    def process_stream(self, cat_feats, item_feats, onsets=None):
        all_res=[]
        for i in range(cat_feats.shape[0]):
            onset = int(onsets[i]) if onsets is not None else i
            trace = self.add_trace(cat_feats[i], item_feats[i], onset_tr=onset)
            all_res.extend(self.run_competition(trace))
        return all_res

    def rsa_matrix(self, use_init=False):
        if not self.traces: return torch.tensor([])
        vecs = torch.stack([t.hidden_init if use_init else t.hidden for t in self.traces])
        v = F.normalize(vecs, dim=1)
        return v @ v.T

    def rsa_change(self):
        return self.rsa_matrix(False) - self.rsa_matrix(True)

    def summary(self):
        dirs=[e['direction'] for e in self.log]; ds=[e['delta_sim'] for e in self.log]
        return dict(n_episodes=len(self.log),
                    n_diff=dirs.count('differentiation'),
                    n_intg=dirs.count('integration'),
                    n_nc=dirs.count('no_change'),
                    mean_delta=float(np.mean(ds)) if ds else 0.0)


## 6. Run across all StudyForrest runs

In [ ]:
run_archives={}; run_results={}

for run in RUNS:
    print(f"Run {run} ...", end=" ")
    arch = MemoryArchive(N_HIDDEN, COMPETITOR_THRESH, MIN_TEMPORAL_GAP,
                         OSC_AMP, OSC_PERIOD, N_OSC_TICKS, LR)
    res  = arch.process_stream(run_data[run]['cat_feats'],
                               run_data[run]['item_feats'],
                               onsets=run_data[run]['onsets'])
    run_archives[run]=arch; run_results[run]=res
    s=arch.summary()
    print(f"{len(arch.traces)} events, {s['n_episodes']} episodes  "
          f"diff={s['n_diff']} intg={s['n_intg']}  mean Δ={s['mean_delta']:+.4f}")


In [ ]:
# ── Auto-print: per-run and overall competition summary ─────────────────────
all_entries = [e for r in RUNS for e in run_results[r]]
total_eps   = len(all_entries)
total_diff  = sum(1 for e in all_entries if e['direction']=='differentiation')
total_intg  = sum(1 for e in all_entries if e['direction']=='integration')
total_nc    = sum(1 for e in all_entries if e['direction']=='no_change')
all_deltas  = [e['delta_sim'] for e in all_entries]
all_comps   = [e['competitor_activity'] for e in all_entries]

print("=" * 65)
print("MEMORY ARCHIVE — COMPETITION RESULTS")
print("=" * 65)
print(f"  {'Run':>4}  {'Events':>6}  {'Episodes':>8}  {'Diff%':>6}  {'Intg%':>6}  {'Mean Δsim':>10}")
print("  " + "-"*61)
for run in RUNS:
    s = run_archives[run].summary()
    n = s['n_episodes']
    pct_d = s['n_diff']/max(n,1)*100
    pct_i = s['n_intg']/max(n,1)*100
    print(f"  {run:>4}  {len(run_archives[run].traces):>6}  {n:>8}  "
          f"{pct_d:>5.1f}%  {pct_i:>5.1f}%  {s['mean_delta']:>+10.4f}")
print()
print(f"  TOTAL  {sum(len(run_archives[r].traces) for r in RUNS):>6}  "
      f"{total_eps:>8}  "
      f"{total_diff/max(total_eps,1)*100:>5.1f}%  "
      f"{total_intg/max(total_eps,1)*100:>5.1f}%  "
      f"{np.mean(all_deltas) if all_deltas else 0:>+10.4f}")
print("=" * 65)
print()

if not all_entries:
    print("⚠  NO competition episodes recorded.")
    print(f"   Current competitor threshold: {COMPETITOR_THRESH}.")
    print("   Try lowering COMPETITOR_THRESH (e.g. to 0.5) or checking feature loading.")
else:
    dom = 'differentiation' if total_diff > total_intg else 'integration'
    print(f"  Dominant outcome: {dom} ({max(total_diff,total_intg)/total_eps*100:.1f}% of episodes)")
    print(f"  Mean competitor activity : {np.mean(all_comps):.4f}")
    print(f"  Mean Δsim overall        : {np.mean(all_deltas):+.4f}")
    print()
    if total_eps < 20:
        print("⚠  Fewer than 20 competition episodes total.")
        print("   Consider lowering COMPETITOR_THRESH or MIN_TEMPORAL_GAP.")
    elif total_diff / max(total_eps,1) > 0.5:
        print("✓  Differentiation dominates — consistent with hippocampal sparse coding.")
        print("   osc_amp may be in the moderate-activity regime.")
    elif total_intg / max(total_eps,1) > 0.5:
        print("ℹ  Integration dominates — osc_amp may be too high (strong reactivation).")
        print("   Try lowering OSC_AMP in the config above.")
    else:
        print("ℹ  Mixed outcomes — both differentiation and integration occurring.")
        print("   This is expected when stimulus similarity varies across the film.")
print("=" * 65)


**Interpreting competition outcomes:**

Each *competition episode* is one instance where a newly encoded event trace finds a
pairmate among prior traces (cosine similarity > `COMPETITOR_THRESH` in the category subspace).
The oscillatory retrieval pass then determines the competitor's reactivation level, which feeds
into the BCM update of the target trace's hidden representation.

**What the numbers mean:**
- **Diff%** — fraction of episodes where the target and competitor traces became *less* similar
  (weakened shared connections; hidden units becoming more exclusive). This maps to the hippocampal
  pattern similarity *decreases* predicted by Ritvo et al. (H1).
- **Intg%** — fraction where traces became *more* similar (strengthened shared connections).
  Expected when two events are very semantically similar and closely repeated.
- **Mean Δsim** — signed average representational change. Negative = net differentiation across
  the film; positive = net integration.

**Across-run variation** is expected: later runs may show more integration as the film is more
familiar (more overlap with existing traces), while earlier runs may show more differentiation
(new events are more surprising and compete with fewer established traces).

**If no episodes are found:** the category features may not have enough overlap across events.
Lower `COMPETITOR_THRESH` or increase `N_CATEGORY` to capture more semantic structure.


## 7. Visualise RSA change matrices

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for i, run in enumerate(RUNS):
    ax = axes[i//4, i%4]
    mat = run_archives[run].rsa_change().numpy()
    im  = ax.imshow(mat, cmap='RdBu_r', vmin=-0.3, vmax=0.3, aspect='auto')
    ax.set_title(f"Run {run}  (N={len(run_archives[run].traces)})", fontsize=10)
    ax.set_xlabel("Event"); ax.set_ylabel("Event")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("RSA change (blue=diff, red=intg)", fontsize=12, y=1.01)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/rsa_change_matrices.png", dpi=150, bbox_inches='tight')
plt.show()


**Reading the RSA change matrices:** Each matrix shows Δsim (post-competition minus pre-competition
cosine similarity) for every pair of events in a run.

- **Blue cells** (negative values) — these event pairs were differentiated: the model weakened
  their shared hidden-layer connections. In the brain, we would predict *reduced* hippocampal
  pattern similarity between these events on later re-exposure (H1).
- **Red cells** (positive values) — integration: shared connections strengthened. Predicted to
  correspond to *increased* hippocampal RSA across runs.
- **White/near-zero cells** — no competition episode occurred (events were not classified as
  pairmates) or the competitor was inactive (below θ_low, no BCM update).

**Diagonal** is always zero (events don't compete with themselves).

**What to look for:** Structured off-diagonal clusters of blue or red indicate events that
co-occur thematically (e.g. repeated scenes, recurring characters) and have undergone systematic
representational change. Random scatter suggests the competitor threshold may be too permissive.


## 8. Competition outcome statistics

In [ ]:
sums = {r: run_archives[r].summary() for r in RUNS}
x = np.arange(len(RUNS)); w = 0.5

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax=axes[0]
totals=[sums[r]['n_episodes'] for r in RUNS]
ax.bar(x, [sums[r]['n_diff']/max(t,1)*100 for r,t in zip(RUNS,totals)], w,
       label='Differentiation', color='steelblue')
ax.bar(x, [sums[r]['n_intg']/max(t,1)*100 for r,t in zip(RUNS,totals)], w,
       bottom=[sums[r]['n_diff']/max(t,1)*100 for r,t in zip(RUNS,totals)],
       label='Integration', color='darkorange')
ax.bar(x, [sums[r]['n_nc']/max(t,1)*100 for r,t in zip(RUNS,totals)], w,
       bottom=[(sums[r]['n_diff']+sums[r]['n_intg'])/max(t,1)*100 for r,t in zip(RUNS,totals)],
       label='No change', color='lightgrey')
ax.set_xticks(x); ax.set_xticklabels([f"R{r}" for r in RUNS])
ax.set_ylabel("% of episodes"); ax.set_title("Outcomes by run"); ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax=axes[1]
deltas=[sums[r]['mean_delta'] for r in RUNS]
ax.bar(x, deltas, w, color=['steelblue' if d<0 else 'darkorange' for d in deltas])
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xticks(x); ax.set_xticklabels([f"R{r}" for r in RUNS])
ax.set_ylabel("Mean Δ sim"); ax.set_title("Mean representational change")
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax=axes[2]
all_d=[e['delta_sim'] for r in RUNS for e in run_results[r]]
ax.hist(all_d, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='k', lw=1.2, ls='--')
ax.axvline(np.mean(all_d) if all_d else 0, color='red', lw=1.5,
           label=f"Mean={np.mean(all_d):+.3f}" if all_d else "no data")
ax.set_xlabel("Δ sim"); ax.set_ylabel("Count"); ax.set_title("All episodes")
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/competition_stats.png", dpi=150, bbox_inches='tight')
plt.show()


## 9. Pairmate scatter: before vs after competition

In [ ]:
all_entries=[e for r in RUNS for e in run_results[r]]
if all_entries:
    sb=np.array([e['sim_before'] for e in all_entries])
    sa=np.array([e['sim_after'] for e in all_entries])
    dirs=[e['direction'] for e in all_entries]
    camp=np.array([e['competitor_activity'] for e in all_entries])
    cmap={'differentiation':'steelblue','integration':'darkorange','no_change':'grey'}
    colours=[cmap[d] for d in dirs]

    fig, axes=plt.subplots(1,2,figsize=(13,5))
    ax=axes[0]
    for d,c in cmap.items():
        m=[x==d for x in dirs]
        ax.scatter(sb[m],sa[m],c=c,alpha=0.5,s=20,label=d.capitalize())
    lm=[min(sb.min(),sa.min())-.05, max(sb.max(),sa.max())+.05]
    ax.plot(lm,lm,'k--',lw=0.8,alpha=0.4)
    ax.set_xlabel("Sim before"); ax.set_ylabel("Sim after")
    ax.set_title("Pairmate RSA: before vs after competition")
    ax.legend(fontsize=9); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    ax=axes[1]
    ax.scatter(camp, sa-sb, c=colours, alpha=0.5, s=20)
    ax.axhline(0,color='k',lw=0.8,ls='--')
    ax.set_xlabel("Competitor activity"); ax.set_ylabel("Δ sim")
    ax.set_title("Competitor activity → direction of change")
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor=c,label=l.capitalize()) for l,c in cmap.items()], fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/pairmate_scatter.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No competition episodes — try lowering COMPETITOR_THRESH.")


In [ ]:
# ── Auto-print: pairmate scatter summary ────────────────────────────────────
if all_entries:
    sb_arr = np.array([e['sim_before'] for e in all_entries])
    sa_arr = np.array([e['sim_after']  for e in all_entries])
    ca_arr = np.array([e['competitor_activity'] for e in all_entries])

    # Spearman correlation: competitor activity → Δsim
    from scipy.stats import spearmanr
    r_ca, p_ca = spearmanr(ca_arr, sa_arr - sb_arr)

    print("=" * 60)
    print("PAIRMATE SCATTER — SUMMARY")
    print("=" * 60)
    print(f"  N competition episodes : {len(all_entries)}")
    print(f"  Mean sim before        : {sb_arr.mean():+.4f}  (SD={sb_arr.std():.4f})")
    print(f"  Mean sim after         : {sa_arr.mean():+.4f}  (SD={sa_arr.std():.4f})")
    print(f"  Mean Δsim              : {(sa_arr-sb_arr).mean():+.4f}")
    print(f"  Competitor activity × Δsim  r = {r_ca:.3f}, p = {p_ca:.4f}")
    print()

    if r_ca > 0.1 and p_ca < 0.05:
        print("✓  Higher competitor activity correlates with more positive Δsim (integration).")
        print("   This is the expected NMPH signature: the U-shaped curve means that as")
        print("   competitor activity crosses into the strengthening zone, Δsim turns positive.")
    elif r_ca < -0.1 and p_ca < 0.05:
        print("ℹ  Higher competitor activity correlates with more negative Δsim.")
        print("   The model may be operating primarily in the weakening zone for this stimulus.")
    else:
        print("ℹ  No significant competitor activity × Δsim relationship detected.")
        print("   This can occur if most episodes fall in a narrow band of competitor activity.")
    print()
    print(f"  Pairs with sim_after < 0 (anticorrelated) : "
          f"{(sa_arr<0).sum()} ({(sa_arr<0).mean()*100:.1f}%)")
    if (sa_arr<0).any():
        print("  → Anticorrelated event pairs detected. These are the strongest candidates")
        print("    for below-zero fMRI hippocampal RSA (see H3 in docs/hypotheses.md).")
    print("=" * 60)


## 10. Save results for notebook 03 (brain comparison)

In [ ]:
import pickle

pkg = {
    'rsa_before': {r: run_archives[r].rsa_matrix(True).numpy() for r in RUNS},
    'rsa_after':  {r: run_archives[r].rsa_matrix(False).numpy() for r in RUNS},
    'rsa_change': {r: run_archives[r].rsa_change().numpy() for r in RUNS},
    'log':        run_results,
    'n_events':   {r: len(run_archives[r].traces) for r in RUNS},
    'onsets':     {r: run_data[r]['onsets'] for r in RUNS},
    'durations':  {r: run_data[r]['durations'] for r in RUNS},
    'config': dict(n_category=N_CATEGORY, competitor_threshold=COMPETITOR_THRESH,
                   min_temporal_gap=MIN_TEMPORAL_GAP, osc_amp=OSC_AMP,
                   n_hidden=N_HIDDEN)
}
save_path = f"{RESULTS_DIR}/nmph_naturalistic_results.pkl"
with open(save_path,'wb') as f: pickle.dump(pkg,f)
for r in RUNS:
    np.save(f"{RESULTS_DIR}/rsa_change_run{r}.npy", pkg['rsa_change'][r])
print(f"Saved -> {save_path}")


## 11. GitHub Sync — Push

In [ ]:
import json as _j, subprocess as _sp
from datetime import datetime as _dt
from google.colab import _message

_nb   = _message.blocking_request('get_ipynb', timeout_sec=30)
_dest = f"{REPO_PATH}/{NOTEBOOK_REL}"
import os as _os; _os.makedirs(_os.path.dirname(_dest), exist_ok=True)
with open(_dest,'w') as _f: _j.dump(_nb,_f,indent=1)
_msg = f"[colab] 02_naturalistic_pipeline: {_dt.now():%Y-%m-%d %H:%M}"
_sp.run(["git","-C",REPO_PATH,"add",NOTEBOOK_REL], check=True)
_res = _sp.run(["git","-C",REPO_PATH,"commit","-m",_msg], capture_output=True, text=True)
if "nothing to commit" in _res.stdout: print("Nothing to commit.")
else:
    _sp.run(["git","-C",REPO_PATH,"push"], check=True)
    print(f"Pushed: {_msg}")
